In [2]:
n_layers = 16
dmodel = 64 * n_layers
n_experts = 1
dff = n_experts * 4 * (dmodel)

In [3]:
attn_params = 4 * (dmodel ** 2)
ff_params = 2 * dmodel * dff
ff_active = 8 * (dmodel ** 2)
embedding = 50_257 * dmodel
model_params = n_layers * (attn_params + ff_params) + 2 * embedding
model_active = n_layers * (attn_params + ff_active) + embedding

print(f"active / total: {model_active / 1e9:.2f}B / {model_params / 1e9:.2f}B")
print(f"active / total: {model_active / 1e6:.2f}M / {model_params / 1e6:.2f}M")


active / total: 0.25B / 0.30B
active / total: 252.79M / 304.25M


In [9]:
sl = 2048
bs = 64
steps = 920_001
# steps = 320_001
tokens = sl * bs * steps
print(f"tokens: {tokens / 1e9:.2f}B")

tokens: 120.59B


In [5]:
time = 0.5 * 60 ** 2

In [6]:
h100 = (1_979 / 2) * 1e12
# a100 = 312 * 1e12
n_gpus = 4

In [7]:
r_flops = time * h100 * n_gpus

print(r_flops)

th_flops = model_active * tokens * 6

mfu = th_flops / r_flops

7.1244e+18


In [8]:
print(f"MFU: {mfu}")

MFU: 0.18695933816087587


In [8]:
def model_params(dmodel, dff, n_blocks):
    attn = 4 * dmodel ** 2
    ff = dmodel * dff * 3
    embeds = 50_304 * dmodel * 2
    params = n_blocks * (attn + ff) + embeds
    return params

In [ ]:
model_normal = {
    "dmodel": 1024,
    "dff": 2560,
    "n_blocks": 16,
}

dmt = 768
dfft = 2.5 * dmt
print(f"dfft: {dfft} = 64 * {dfft / 64}")
model_thin = {
    "dmodel": dmt,
    "dff": dfft,
    "n_blocks": 32,
}

dmf = 1344
dfff = 2.5 * dmf - 128 - 32
print(f"dfff: {dfff} = 64 * {dfff / 64}")
model_fat = {
    "dmodel": dmf,
    "dff": dfff,
    "n_blocks": 8,
}


print(f"model_normal: {model_params(**model_normal)/1e6:.2f} M")
print(f"model_thin: {model_params(**model_thin)/1e6:.2f} M")
print(f"model_fat: {model_params(**model_fat)/1e6:.2f} M")

dfft: 1920.0 = 64 * 30.0
dfff: 3200.0 = 64 * 50.0
model_normal: 295.96 M
model_thin: 294.32 M
model_fat: 296.24 M


In [46]:
print(model_fat)

{'dmodel': 1344, 'dff': 3200.0, 'n_blocks': 8}


In [1]:
24 * 64

1536